# Object-Oriented Programming (OOP) in Python
a CLASS is a blueprint ; an OBJECT (instance) is a thing built from that blueprint.
an object keeps its DATA (attributes) and its BEHAVIOUR (methods) together.

    class      ---> Car (the design)
    object     ---> my_car , your_car (real cars built from the design)
    attribute  ---> color , speed           (data)
    method     ---> accelerate() , brake()  (behaviour)

the 4 pillars of OOP :

    1. Encapsulation   ---> keep data and the code that uses it together , hide internal details
    2. Abstraction     ---> show a simple interface (fit / predict) , hide the complexity
    3. Inheritance     ---> build a new class from an existing one
    4. Polymorphism    ---> different classes , same interface , used in the same way

you already used objects everywhere : list.append() , str.upper() , df.head() , model.fit()

## 1. From dictionaries to classes

In [1]:
# without classes : data in a dict , functions outside --- nothing connects them
account = {"owner": "Maryam", "balance": 1000}

def deposit(acc, amount):
    acc["balance"] += amount

deposit(account, 500)
account["balance"] = -99999          # anyone can break the data , no protection
print(account)

{'owner': 'Maryam', 'balance': -99999}


In [2]:
# with a class : data + behaviour together
class BankAccount:
    """A simple bank account."""

    def __init__(self, owner, balance=0):     # constructor : runs when an object is created
        self.owner = owner                     # instance attributes
        self.balance = balance

    def deposit(self, amount):                 # a method : a function that belongs to the class
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount
        return self.balance

acc1 = BankAccount("Maryam", 1000)             # create an object (instance)
acc2 = BankAccount("Ali")
acc1.deposit(500)
print(acc1.owner, acc1.balance)
print(acc2.owner, acc2.balance)
print(type(acc1), isinstance(acc1, BankAccount))

Maryam 1500
Ali 0
<class '__main__.BankAccount'> True


In [3]:
# self is the object itself :  acc1.deposit(200)  is the same as  BankAccount.deposit(acc1, 200)
BankAccount.deposit(acc1, 200)
print(acc1.balance)

# each object has its OWN attributes (stored in __dict__)
print(acc1.__dict__)
print(acc2.__dict__)
print(acc1 is acc2)

1700
{'owner': 'Maryam', 'balance': 1700}
{'owner': 'Ali', 'balance': 0}
False


## 2. Instance attributes vs class attributes

In [4]:
class Sensor:
    unit = "celsius"          # CLASS attribute : shared by ALL sensors
    count = 0

    def __init__(self, sensor_id, value):
        self.sensor_id = sensor_id      # INSTANCE attributes : different for each sensor
        self.value = value
        Sensor.count += 1               # change the shared counter

s1 = Sensor("temp_01", 72.5)
s2 = Sensor("temp_02", 68.0)
print(s1.unit, s2.unit, Sensor.unit)
print("sensors created:", Sensor.count)

celsius celsius celsius
sensors created: 2


In [5]:
# PITFALL : a MUTABLE class attribute is shared by all objects
class BadModel:
    history = []                        # ONE list for ALL objects !
    def log(self, loss):
        self.history.append(loss)

m1, m2 = BadModel(), BadModel()
m1.log(0.9)
print("m2 history:", m2.history)        # m2 sees m1's loss !

class GoodModel:
    def __init__(self):
        self.history = []               # a NEW list for each object
    def log(self, loss):
        self.history.append(loss)

g1, g2 = GoodModel(), GoodModel()
g1.log(0.9)
print("g2 history:", g2.history)

m2 history: [0.9]
g2 history: []


## 3. Special (dunder) methods : make your objects behave like built-in types

In [6]:
# __repr__ (for developers) and __str__ (for users)
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"
    def __str__(self):
        return f"({self.x}, {self.y})"

p = Point(3, 4)
print(p)              # uses __str__
print(repr(p))        # uses __repr__
print([p, Point(0, 0)])   # containers use __repr__

(3, 4)
Point(x=3, y=4)
[Point(x=3, y=4), Point(x=0, y=0)]


In [7]:
# arithmetic and comparison : __add__ , __mul__ , __eq__ , __abs__
import math

class Vector:
    def __init__(self, *values):
        self.values = tuple(values)
    def __repr__(self):
        return f"Vector{self.values}"
    def __add__(self, other):
        return Vector(*(a + b for a, b in zip(self.values, other.values, strict=True)))
    def __mul__(self, scalar):
        return Vector(*(a * scalar for a in self.values))
    def __rmul__(self, scalar):                 # makes  2 * v  work (not only v * 2)
        return self * scalar
    def __matmul__(self, other):                # dot product with @
        return sum(a * b for a, b in zip(self.values, other.values, strict=True))
    def __abs__(self):                          # length (norm)
        return math.sqrt(self @ self)
    def __eq__(self, other):
        return isinstance(other, Vector) and self.values == other.values
    def __len__(self):
        return len(self.values)

v, w = Vector(1, 2, 3), Vector(4, 5, 6)
print(v + w, v * 2, 2 * v)
print("dot:", v @ w, "| length:", round(abs(Vector(3, 4)), 2), "| len:", len(v))
print(v == Vector(1, 2, 3), v == w)

Vector(5, 7, 9) Vector(2, 4, 6) Vector(2, 4, 6)
dot: 32 | length: 5.0 | len: 3
True False


In [8]:
# __len__ , __getitem__ , __iter__ , __contains__ : behave like a sequence
class Vocabulary:
    def __init__(self, tokens):
        self.itos = ["<pad>", "<unk>"] + sorted(set(tokens))     # index ---> string
        self.stoi = {t: i for i, t in enumerate(self.itos)}       # string ---> index
    def __len__(self):
        return len(self.itos)
    def __getitem__(self, token):
        return self.stoi.get(token, 1)                            # unknown ---> <unk>
    def __contains__(self, token):
        return token in self.stoi
    def __iter__(self):
        return iter(self.itos)
    def encode(self, text):
        return [self[t] for t in text.lower().split()]

vocab = Vocabulary("the cat sat on the mat".split())
print(len(vocab), list(vocab))
print(vocab["cat"], vocab["dog"], "cat" in vocab, "dog" in vocab)
print(vocab.encode("The dog sat on the cat"))

7 ['<pad>', '<unk>', 'cat', 'mat', 'on', 'sat', 'the']
2 1 True False
[6, 1, 5, 4, 6, 2]


In [9]:
# __call__ : an object that can be CALLED like a function (layers , models , scalers)
class Threshold:
    def __init__(self, t):
        self.t = t
    def __call__(self, prob):
        return int(prob >= self.t)

strict = Threshold(0.9)
print(strict(0.95), strict(0.5), callable(strict))
print(list(map(Threshold(0.5), [0.2, 0.7, 0.5])))

1 0 True
[0, 1, 1]


In [10]:
# __lt__ : sorting objects ;  __bool__ : truthiness
class Result:
    def __init__(self, name, score):
        self.name, self.score = name, score
    def __repr__(self):
        return f"{self.name}:{self.score}"
    def __lt__(self, other):
        return self.score < other.score
    def __bool__(self):
        return self.score >= 0.8              # 'good enough' result

results = [Result("rf", 0.91), Result("lr", 0.78), Result("xgb", 0.93)]
print(sorted(results), max(results))
print([r.name for r in results if r])

[lr:0.78, rf:0.91, xgb:0.93] xgb:0.93
['rf', 'xgb']


## 4. Encapsulation : protect the data

In [11]:
# naming conventions :
#   name     ---> public
#   _name    ---> 'internal' (a convention : please don't touch from outside)
#   __name   ---> name mangling (harder to access by accident)
class Account:
    def __init__(self, balance):
        self._balance = balance
        self.__pin = "1234"

a = Account(100)
print(a._balance)                 # possible , but a signal that you should not
try:
    print(a.__pin)
except AttributeError as e:
    print("AttributeError:", e)
print(a._Account__pin)            # mangled name : still reachable , Python has no real 'private'

100
AttributeError: 'Account' object has no attribute '__pin'
1234


In [12]:
# @property : attribute-like access with validation behind it
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius            # goes through the setter below

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError(f"below absolute zero: {value}")
        self._celsius = value

    @property
    def fahrenheit(self):                 # computed , read-only
        return self._celsius * 9 / 5 + 32

t = Temperature(25)
print(t.celsius, t.fahrenheit)
t.celsius = 30
print(t.celsius, t.fahrenheit)
for bad in (lambda: setattr(t, "celsius", -300), lambda: setattr(t, "fahrenheit", 100)):
    try:
        bad()
    except (ValueError, AttributeError) as e:
        print(type(e).__name__, ":", e)

25 77.0
30 86.0
ValueError : below absolute zero: -300
AttributeError : property 'fahrenheit' of 'Temperature' object has no setter


## 5. @classmethod and @staticmethod

In [13]:
import json

class ModelConfig:
    VALID_OPTIMIZERS = {"sgd", "adam"}

    def __init__(self, lr, epochs, optimizer="adam"):
        if not ModelConfig.is_valid_lr(lr):
            raise ValueError(f"invalid lr: {lr}")
        if optimizer not in self.VALID_OPTIMIZERS:
            raise ValueError(f"unknown optimizer: {optimizer!r}")
        self.lr, self.epochs, self.optimizer = lr, epochs, optimizer

    def __repr__(self):
        return f"ModelConfig(lr={self.lr}, epochs={self.epochs}, optimizer={self.optimizer!r})"

    @classmethod
    def from_dict(cls, d):                     # ALTERNATIVE constructor : gets the CLASS (cls)
        return cls(**d)

    @classmethod
    def from_json(cls, text):
        return cls.from_dict(json.loads(text))

    @staticmethod
    def is_valid_lr(lr):                       # a plain function that lives in the class (no self , no cls)
        return isinstance(lr, (int, float)) and 0 < lr < 1

print(ModelConfig(0.001, 10))
print(ModelConfig.from_dict({"lr": 0.01, "epochs": 5, "optimizer": "sgd"}))
print(ModelConfig.from_json('{"lr": 0.0003, "epochs": 30}'))
print(ModelConfig.is_valid_lr(5))

ModelConfig(lr=0.001, epochs=10, optimizer='adam')
ModelConfig(lr=0.01, epochs=5, optimizer='sgd')
ModelConfig(lr=0.0003, epochs=30, optimizer='adam')
False


## 6. Inheritance

In [14]:
class Employee:
    raise_rate = 1.05
    def __init__(self, name, salary):
        self.name, self.salary = name, salary
    def describe(self):
        return f"{self.name} earns {self.salary:,}"
    def apply_raise(self):
        self.salary = int(self.salary * self.raise_rate)

class DataScientist(Employee):               # DataScientist IS-A Employee
    raise_rate = 1.10                        # override a class attribute
    def __init__(self, name, salary, languages):
        super().__init__(name, salary)       # reuse the parent's __init__
        self.languages = languages
    def describe(self):                      # override a method and extend it
        return super().describe() + f" and codes in {', '.join(self.languages)}"

e = Employee("Sara", 40_000)
d = DataScientist("Maryam", 50_000, ["Python", "SQL"])
for person in (e, d):
    person.apply_raise()                     # inherited method , but uses each class's raise_rate
    print(person.describe())
print(isinstance(d, Employee), issubclass(DataScientist, Employee), isinstance(e, DataScientist))
print([c.__name__ for c in DataScientist.__mro__])

Sara earns 42,000
Maryam earns 55,000 and codes in Python, SQL
True True False
['DataScientist', 'Employee', 'object']


In [15]:
# abstract base class : defines an INTERFACE that every subclass MUST implement
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        ...
    def describe(self):                       # a normal method that uses the abstract one
        return f"{type(self).__name__} with area {self.area():.2f}"

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r ** 2

class Rectangle(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h

try:
    Shape()
except TypeError as e:
    print("TypeError:", e)

class Broken(Shape):
    pass
try:
    Broken()
except TypeError as e:
    print("TypeError:", e)

TypeError: Can't instantiate abstract class Shape without an implementation for abstract method 'area'
TypeError: Can't instantiate abstract class Broken without an implementation for abstract method 'area'


## 7. Polymorphism : same interface , different behaviour

In [16]:
shapes = [Circle(1), Rectangle(2, 3), Circle(2.5)]
for s in shapes:
    print(s.describe())                       # the SAME call works for every shape
print("total area:", round(sum(s.area() for s in shapes), 2))
print("largest   :", max(shapes, key=lambda s: s.area()).describe())

Circle with area 3.14
Rectangle with area 6.00
Circle with area 19.63
total area: 28.78
largest   : Circle with area 19.63


In [17]:
# duck typing : no common parent is needed , only the same method name
class Dog:
    def speak(self): return "woof"
class Robot:
    def speak(self): return "beep"

for thing in (Dog(), Robot()):
    print(type(thing).__name__, "says", thing.speak())

Dog says woof
Robot says beep


## 8. Composition : HAS-A instead of IS-A

In [18]:
class Engine:
    def __init__(self, hp): self.hp = hp
    def start(self): return f"engine {self.hp}hp started"

class Car:
    def __init__(self, model, hp):
        self.model = model
        self.engine = Engine(hp)              # a Car HAS-AN Engine
    def drive(self):
        return f"{self.model}: {self.engine.start()} , driving"

print(Car("Tara", 110).drive())

Tara: engine 110hp started , driving


## 9. dataclasses : classes for data with less code

In [19]:
from dataclasses import dataclass, field, asdict

@dataclass
class Experiment:
    name: str
    lr: float = 0.001
    epochs: int = 10
    tags: list[str] = field(default_factory=list)     # a NEW list for each object (no shared default)

    def __post_init__(self):                           # validation after the auto-generated __init__
        if not 0 < self.lr < 1:
            raise ValueError(f"lr must be in (0, 1), got {self.lr}")

e1 = Experiment("baseline")
e2 = Experiment("bigger_lr", lr=0.01, tags=["sweep"])
e1.tags.append("v1")
print(e1)                       # automatic __repr__
print(e2)
print(e1 == Experiment("baseline", tags=["v1"]))      # automatic __eq__
print(asdict(e2))               # ---> dict (easy to save as JSON)

Experiment(name='baseline', lr=0.001, epochs=10, tags=['v1'])
Experiment(name='bigger_lr', lr=0.01, epochs=10, tags=['sweep'])
True
{'name': 'bigger_lr', 'lr': 0.01, 'epochs': 10, 'tags': ['sweep']}


In [20]:
# frozen=True : immutable and hashable ;  order=True : comparison operators
@dataclass(frozen=True, order=True)
class Version:
    major: int
    minor: int
    patch: int = 0

v1, v2 = Version(3, 9), Version(3, 12, 1)
print(v1 < v2, sorted([v2, v1]))
print({v1: "old", v2: "new"}[Version(3, 12, 1)])     # hashable ---> dict key
try:
    v1.major = 4
except Exception as e:
    print(type(e).__name__, ":", e)

True [Version(major=3, minor=9, patch=0), Version(major=3, minor=12, patch=1)]
new
FrozenInstanceError : cannot assign to field 'major'


## 10. Classes that work with  with  and  for

In [21]:
# a context manager class : __enter__ and __exit__
import time
class Timer:
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self.start = time.perf_counter()
        return self                                # the object bound by  as
    def __exit__(self, exc_type, exc, tb):
        self.ms = (time.perf_counter() - self.start) * 1000
        print(f"[{self.label}] {'failed' if exc_type else 'ok'} in {self.ms:.1f} ms")
        return False                               # False ---> do NOT hide exceptions

with Timer("squares") as t:
    total = sum(i * i for i in range(300_000))
print(total, "| elapsed is stored:", t.ms > 0)

try:
    with Timer("division"):
        1 / 0
except ZeroDivisionError:
    print("the exception still reached us")

[squares] ok in 25.5 ms
8999955000050000 | elapsed is stored: True
[division] failed in 0.0 ms
the exception still reached us


In [22]:
# an iterable class : __iter__ with yield (a generator method)
class Countdown:
    def __init__(self, start):
        self.start = start
    def __iter__(self):
        n = self.start
        while n > 0:
            yield n
            n -= 1

c = Countdown(5)
print(list(c), list(c))                # iterable AGAIN : each for-loop gets a fresh generator

[5, 4, 3, 2, 1] [5, 4, 3, 2, 1]


## 11. Applications in AI , Machine Learning and Expert Systems

In [23]:
# Application 1 : a StandardScaler with the scikit-learn API (fit / transform / fit_transform)
class NotFittedError(Exception):
    pass

class StandardScaler:
    def __init__(self, eps=1e-12):
        self.eps = eps                          # hyper-parameter : set in __init__

    def fit(self, X):
        n, d = len(X), len(X[0])
        self.mean_ = [sum(row[j] for row in X) / n for j in range(d)]           # learned : trailing _
        self.scale_ = [
            (sum((row[j] - self.mean_[j]) ** 2 for row in X) / n) ** 0.5 for j in range(d)
        ]
        return self                             # return self ---> method chaining

    def transform(self, X):
        if not hasattr(self, "mean_"):
            raise NotFittedError("call fit() before transform()")
        return [[(x - m) / (s + self.eps) for x, m, s in zip(row, self.mean_, self.scale_)] for row in X]

    def fit_transform(self, X):
        return self.fit(X).transform(X)

X_train = [[20, 30_000], [30, 50_000], [40, 70_000]]
X_test = [[35, 60_000]]
scaler = StandardScaler()
try:
    scaler.transform(X_test)
except NotFittedError as e:
    print("NotFittedError:", e)
print([[round(v, 3) for v in row] for row in scaler.fit_transform(X_train)])
print("learned mean:", scaler.mean_)
print("test (train statistics):", [[round(v, 3) for v in row] for row in scaler.transform(X_test)])

NotFittedError: call fit() before transform()
[[-1.225, -1.225], [0.0, 0.0], [1.225, 1.225]]
learned mean: [30.0, 50000.0]
test (train statistics): [[0.612, 0.612]]


In [24]:
# Application 2 : a base class for models + two very different models with the SAME interface
class BaseModel(ABC):
    @abstractmethod
    def fit(self, X, y): ...
    @abstractmethod
    def predict(self, X): ...

    def score(self, X, y):                      # shared by ALL models (written once)
        preds = self.predict(X)
        return sum(p == t for p, t in zip(preds, y, strict=True)) / len(y)

    def __repr__(self):
        params = ", ".join(f"{k}={v}" for k, v in vars(self).items() if not k.endswith("_"))
        return f"{type(self).__name__}({params})"

class MajorityClassifier(BaseModel):
    """Baseline : always predicts the most common class."""
    def fit(self, X, y):
        from collections import Counter
        self.majority_ = Counter(y).most_common(1)[0][0]
        return self
    def predict(self, X):
        return [self.majority_ for _ in X]

class KNNClassifier(BaseModel):
    """k-nearest neighbours : vote of the k closest training points."""
    def __init__(self, k=3):
        self.k = k
    def fit(self, X, y):
        self.X_, self.y_ = X, y                 # 'lazy learner' : just remember the data
        return self
    def predict(self, X):
        from collections import Counter
        preds = []
        for x in X:
            dists = sorted((math.dist(x, xi), yi) for xi, yi in zip(self.X_, self.y_))
            votes = Counter(label for _, label in dists[: self.k])
            preds.append(votes.most_common(1)[0][0])
        return preds

In [25]:
# a tiny 2D dataset : class 1 when the point is far from the origin
import random
random.seed(1)
def make_data(n):
    X = [[random.uniform(-3, 3), random.uniform(-3, 3)] for _ in range(n)]
    y = [int(math.hypot(a, b) > 2) for a, b in X]
    return X, y

X_tr, y_tr = make_data(120)
X_te, y_te = make_data(60)

for model in [MajorityClassifier(), KNNClassifier(k=1), KNNClassifier(k=5)]:
    model.fit(X_tr, y_tr)                       # polymorphism : the same calls for every model
    print(f"{model!r:<28} test accuracy = {model.score(X_te, y_te):.3f}")

MajorityClassifier()         test accuracy = 0.783
KNNClassifier(k=1)           test accuracy = 0.967
KNNClassifier(k=5)           test accuracy = 0.983


In [26]:
# Application 3 : a Pipeline built by COMPOSITION (it HAS steps) and it is itself a model (IS-A BaseModel)
class Pipeline(BaseModel):
    def __init__(self, steps):
        self.steps = steps                      # list of (name, object)
    def fit(self, X, y):
        for name, step in self.steps[:-1]:
            X = step.fit_transform(X)           # transformers : fit on TRAIN data only
        self.steps[-1][1].fit(X, y)             # the last step is the model
        return self
    def predict(self, X):
        for name, step in self.steps[:-1]:
            X = step.transform(X)               # reuse the training statistics
        return self.steps[-1][1].predict(X)
    def __repr__(self):
        return " -> ".join(name for name, _ in self.steps)

# make one feature 1000 times larger : distances are dominated by it without scaling
X_tr_big = [[a * 1000, b] for a, b in X_tr]
X_te_big = [[a * 1000, b] for a, b in X_te]

raw = KNNClassifier(k=5).fit(X_tr_big, y_tr)
pipe = Pipeline([("scaler", StandardScaler()), ("knn", KNNClassifier(k=5))]).fit(X_tr_big, y_tr)
print(f"KNN without scaling : {raw.score(X_te_big, y_te):.3f}")
print(f"{pipe!r:<20}: {pipe.score(X_te_big, y_te):.3f}")

KNN without scaling : 0.817
scaler -> knn       : 0.983


In [27]:
# Application 4 : a Dataset class + a simple batch loader (the PyTorch idea)
class SensorDataset:
    def __init__(self, readings, labels, transform=None):
        assert len(readings) == len(labels)
        self.readings, self.labels, self.transform = readings, labels, transform
    def __len__(self):
        return len(self.readings)
    def __getitem__(self, i):
        x = self.readings[i]
        if self.transform:
            x = self.transform(x)
        return x, self.labels[i]

class Loader:
    def __init__(self, dataset, batch_size, shuffle=False, seed=0):
        self.dataset, self.batch_size, self.shuffle, self.seed = dataset, batch_size, shuffle, seed
    def __len__(self):
        return -(-len(self.dataset) // self.batch_size)          # ceiling division
    def __iter__(self):
        idx = list(range(len(self.dataset)))
        if self.shuffle:
            random.Random(self.seed).shuffle(idx)
        for start in range(0, len(idx), self.batch_size):
            batch = [self.dataset[i] for i in idx[start:start + self.batch_size]]
            xs, ys = zip(*batch)                                 # unzip
            yield list(xs), list(ys)

ds = SensorDataset([70, 72, 95, 68, 99, 71, 93], [0, 0, 1, 0, 1, 0, 1], transform=lambda t: round((t - 32) * 5 / 9, 1))
print("len:", len(ds), "| item 2:", ds[2])
loader = Loader(ds, batch_size=3, shuffle=True)
print("batches per epoch:", len(loader))
for xb, yb in loader:
    print(xb, yb)

len: 7 | item 2: (35.0, 1)
batches per epoch: 3
[37.2, 35.0, 22.2] [1, 1, 0]
[21.1, 21.7, 20.0] [0, 0, 0]
[33.9] [1]


In [28]:
# Application 5 : Trainer + Callbacks with inheritance (the Keras / Lightning idea)
class Callback:
    def on_epoch_end(self, trainer, epoch, logs):     # default : do nothing
        pass

class PrintLogger(Callback):
    def on_epoch_end(self, trainer, epoch, logs):
        print(f"  epoch {epoch}: " + "  ".join(f"{k}={v:.2f}" for k, v in logs.items()))

class EarlyStopping(Callback):
    def __init__(self, patience=2):
        self.patience, self.best, self.wait = patience, float("inf"), 0
    def on_epoch_end(self, trainer, epoch, logs):
        if logs["val_loss"] < self.best:
            self.best, self.wait = logs["val_loss"], 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                trainer.stop_training = True
                print(f"  early stopping (best val_loss={self.best:.2f})")

class Trainer:
    def __init__(self, callbacks=()):
        self.callbacks = list(callbacks)
        self.stop_training = False
    def fit(self, val_losses):
        for epoch, loss in enumerate(val_losses, start=1):
            logs = {"val_loss": loss}
            for cb in self.callbacks:
                cb.on_epoch_end(self, epoch, logs)
            if self.stop_training:
                break
        return epoch

last = Trainer([PrintLogger(), EarlyStopping(patience=2)]).fit([0.9, 0.7, 0.65, 0.66, 0.67, 0.64])
print("stopped after epoch", last)

  epoch 1: val_loss=0.90
  epoch 2: val_loss=0.70
  epoch 3: val_loss=0.65
  epoch 4: val_loss=0.66
  epoch 5: val_loss=0.67
  early stopping (best val_loss=0.65)
stopped after epoch 5


In [29]:
# Application 6 : an expert system with classes : Rule , KnowledgeBase , InferenceEngine (+ explanations)
@dataclass(frozen=True)
class Rule:
    name: str
    conditions: frozenset
    conclusion: str

class KnowledgeBase:
    def __init__(self):
        self.rules: list[Rule] = []
    def add(self, name, conditions, conclusion):
        self.rules.append(Rule(name, frozenset(conditions), conclusion))
        return self                                   # chaining
    def __len__(self):
        return len(self.rules)

class InferenceEngine:
    def __init__(self, kb: KnowledgeBase):
        self.kb = kb
    def run(self, facts):
        facts = set(facts)
        self.trace = []                               # remember WHY each fact was derived
        changed = True
        while changed:
            changed = False
            for rule in self.kb.rules:
                if rule.conditions <= facts and rule.conclusion not in facts:
                    facts.add(rule.conclusion)
                    self.trace.append(rule)
                    changed = True
        return facts
    def explain(self, fact):
        for rule in self.trace:
            if rule.conclusion == fact:
                return f"{fact}  BECAUSE  {' and '.join(sorted(rule.conditions))}  (rule {rule.name})"
        return f"{fact} was given as a fact"

kb = (KnowledgeBase()
      .add("R1", {"high_temp", "high_vibration"}, "bearing_wear")
      .add("R2", {"bearing_wear", "noise"}, "bearing_failure")
      .add("R3", {"bearing_failure"}, "stop_machine"))
engine = InferenceEngine(kb)
result = engine.run({"high_temp", "high_vibration", "noise"})
print(len(kb), "rules | derived:", sorted(result - {"high_temp", "high_vibration", "noise"}))
for f in ["stop_machine", "bearing_failure", "bearing_wear", "noise"]:
    print(" ", engine.explain(f))

3 rules | derived: ['bearing_failure', 'bearing_wear', 'stop_machine']
  stop_machine  BECAUSE  bearing_failure  (rule R3)
  bearing_failure  BECAUSE  bearing_wear and noise  (rule R2)
  bearing_wear  BECAUSE  high_temp and high_vibration  (rule R1)
  noise was given as a fact


## Exercises
1. add a `withdraw(amount)` method to `BankAccount` that raises a custom `InsufficientFunds` exception.
2. make `balance` a read-only `@property` in `BankAccount` (only `deposit` / `withdraw` can change it).
3. add `__sub__` and `__neg__` to `Vector` , and a `normalize()` method that returns a vector of length 1.
4. write a `MinMaxScaler` class with the same `fit / transform / fit_transform` API , and use it inside `Pipeline`.
5. write a `LinearRegressionGD(BaseModel)` with `fit` (gradient descent , session Loops) , `predict` , and override `score` to return R².
6. add a `ModelCheckpoint(Callback)` that prints "saving" whenever `val_loss` improves.
7. add a `why_not(fact)` method to `InferenceEngine` that lists the missing conditions of the rules that could produce `fact`.
8. when should you use inheritance , and when composition ? give one example of each from machine learning.